# Load environment variables

In [2]:
# Load .env variables
library(dotenv)
load_dot_env(file = ".env")

LOG2CPM <- Sys.getenv("LOG2CPM")
SAMPLES_INFO <- Sys.getenv("SAMPLES_INFO")
GENE_INFO <- Sys.getenv("GENE_INFO")
METADATACROSSECTIONAL <- Sys.getenv("METADATACROSSECTIONAL")
ROSMAP_EXPR_AC <- Sys.getenv("ROSMAP_EXPR_AC")
ROSMAP_EXPR_MF <- Sys.getenv("ROSMAP_EXPR_MF")
ROSMAP_EXPR_PCG <- Sys.getenv("ROSMAP_EXPR_PCG")


In [1]:
# Load .env variables
library(dotenv)
load_dot_env(file = ".env")

# Access environment variables
ADJMAT <- Sys.getenv("ADJMAT")

In [2]:
ADJMAT

[1] "/media/psylab-6028/DATA/Eden/CoExpression_ReProduction/adj_drop__.csv"

In [3]:
devtools::load_all()

ℹ Loading speakeasyR


In [4]:
library(data.table)

In [5]:
expr <- fread(ADJMAT, header = TRUE)


In [6]:
expr <- as.data.frame(expr)

In [7]:
rownames(expr) <- expr[[1]]

In [8]:
expr[[1]] <- NULL

In [9]:
dim(expr)

[1] 51918 51918

In [10]:
expr_matrix <- as.matrix(expr)

In [ ]:
expr <- read.csv(LOG2CPM, row.names = 1, sep = "\t")
dim(expr)

In [ ]:
expr <- read.csv(LOG2CPM, row.names = 1, sep = "\t")
expr_matrix <- as.matrix(expr)

In [19]:
source("/Users/edeneldar/speakeasyR/R/cluster_genes.R")
source("/Users/edeneldar/speakeasyR/R/cluster.R")
source("/Users/edeneldar/speakeasyR/R/knn_graph.R")
source("/Users/edeneldar/speakeasyR/R/order_nodes.R")
source("/Users/edeneldar/speakeasyR/R/speakeasyR-package.R")
source("/Users/edeneldar/speakeasyR/R/utils.R")

Warning message in file(filename, "r", encoding = encoding):
“cannot open file '/Users/edeneldar/speakeasyR/R/cluster_genes.R': No such file or directory”


ERROR: Error in file(filename, "r", encoding = encoding): cannot open the connection


In [20]:
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/cluster_genes.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/cluster.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/knn_graph.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/order_nodes.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/speakeasyR-package.R")
source("/media/psylab-6028/DATA/Eden/speakeasyR/R/utils.R")

In [13]:
expr_data_top1000Var <- expr_matrix[order(apply(expr_matrix, 1, var), decreasing = TRUE)[1:1000], ]

In [14]:
res <- cluster_genes(gene_expression = expr_data_top1000Var, subcluster = 3, min_clust = 30, independent_runs = 10, target_partitions = 20)

In [15]:
res

12,12,5,5,5,5,5,5,5,12,...,5,12,12,5,12,12,12,5,12,12
42,42,15,15,5,15,15,15,15,36,...,22,36,36,22,42,40,42,15,42,36
98,104,18,18,5,24,18,24,18,74,...,52,86,70,50,108,92,108,17,100,74


In [7]:
gcn <- stats::cor(t(expr))


In [8]:
anyNA(gcn)

[1] TRUE

In [21]:
is_directed <- FALSE
discard_transient <- 3
independent_runs <- 10
max_threads <- 0
seed <- 0
target_clusters <- 0
target_partitions <- 20
subcluster <- 3
min_clust <- 30
verbose <- FALSE

ERROR: Error in dtype(gcn): could not find function "dtype"


In [16]:
# Check for invalid values
sum(is.na(gcn))        # Count NA
sum(is.nan(gcn))       # Count NaN
sum(is.infinite(gcn))  # Count Inf

# Replace invalid values with 0
gcn[is.na(gcn)] <- 0
gcn[is.nan(gcn)] <- 0
gcn[is.infinite(gcn)] <- 0

[1] 299497636

[1] 0

[1] 0

In [23]:
expr_matrix[1:5, 1:5]

,AC_ENSG00000000003.14,AC_ENSG00000000419.12,AC_ENSG00000000457.13,AC_ENSG00000000460.16,AC_ENSG00000000938.12
AC_ENSG00000000003.14,1.000000e+00,1.148588e-02,4.194853e-04,9.910762e-06,5.534821e-04
AC_ENSG00000000419.12,1.148588e-02,1.000000e+00,6.550656e-03,1.395688e-05,8.605881e-05
AC_ENSG00000000457.13,4.194853e-04,6.550656e-03,1.000000e+00,5.403395e-03,5.086677e-07
AC_ENSG00000000460.16,9.910762e-06,1.395688e-05,5.403395e-03,1.000000e+00,5.851949e-05
AC_ENSG00000000938.12,5.534821e-04,8.605881e-05,5.086677e-07,5.851949e-05,1.000000e+00


In [11]:
library(RcppAnnoy)

build_knn <- function(mat, k = 50, metric = "Angular") {
  mat <- t(mat)              # genes in rows? adjust as needed
  n <- nrow(mat)
  d <- ncol(mat)
  idx <- new(AnnoyAngular, d)
  for (i in seq_len(n)) idx$addItem(i - 1, mat[i, ])
  idx$build(50)
  neigh_i <- integer(n * k)
  neigh_j <- integer(n * k)
  neigh_w <- numeric(n * k)
  pos <- 1L
  for (i in seq_len(n)) {
    nn <- idx$getNNsByItem(i - 1, k + 1)  # includes self
    nn <- nn[nn != (i - 1)]
    nn <- head(nn, k)
    m <- length(nn)
    if (m) {
      rng <- pos:(pos + m - 1L)
      neigh_i[rng] <- i
      neigh_j[rng] <- nn + 1L
      # Optionally compute correlation only for these pairs:
      xi <- mat[i, ]
      neigh_w[rng] <- vapply(nn, function(j) cor(xi, mat[j + 1L, ]), numeric(1))
      pos <- pos + m
    }
  }
  data.frame(
    from = neigh_i[seq_len(pos - 1L)],
    to   = neigh_j[seq_len(pos - 1L)],
    weight = neigh_w[seq_len(pos - 1L)],
    row.names = NULL
  )
}

In [14]:
knnMatrix <- build_knn(expr_matrix, k = 2, metric = "Angular")

In [15]:
# Determine number of nodes
n_nodes <- max(graph_edges$from, graph_edges$to)
# Build symmetric (if undirected)
if (!is_directed) {
    graph_edges <- rbind(
    graph_edges,
    data.frame(from = graph_edges$to,
                to   = graph_edges$from,
                weight = graph_edges$weight)
    )
}
# Build sparse adjacency
if (!requireNamespace("Matrix", quietly = TRUE)) {
    stop("Please install Matrix package")
}
A <- Matrix::sparseMatrix(i = graph_edges$from,
                            j = graph_edges$to,
                            x = graph_edges$weight,
                            dims = c(n_nodes, n_nodes),
                            symmetric = !is_directed)

ERROR: Error: object 'graph_edges' not found


In [ ]:
adjacency <- A


In [25]:
expr_matrix_knn <- speakeasyR::knn_graph(expr_matrix, k = 2, weighted = TRUE)

ERROR: Error in speakeasyR::knn_graph(expr_matrix, k = 2, weighted = TRUE): long vectors (argument 1) are not supported in .C


In [ ]:
results <- speakeasyR::cluster(adjacency,
    discard_transient = discard_transient,
    independent_runs = independent_runs, max_threads = max_threads, seed = seed,
    target_clusters = target_clusters, target_partitions = target_partitions,
    subcluster = subcluster, min_clust = min_clust, verbose = verbose,
    is_directed = is_directed
  )

ERROR: Error in speakeasyR::cluster(expr_matrix, discard_transient = discard_transient, : long vectors (argument 3) are not supported in .C
